## 09.01 门控循环单元（GRU）


### 环境配置


In [1]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torch_npu
    import logging

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor")
logging.getLogger("torch_npu").setLevel(logging.WARNING)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

### 练习 9.1.1

**题目：** 假设我们只想使用时间步$t'$的输入来预测时间步$t > t'$的输出。对于每个时间步，重置门和更新门的最佳值是什么？

**解答：**

如果只想采用时间步$t'$的输出来预测时间步$t>t'$的输出，则模型需要保留过去时间步$t'$的信息，同时忽略当前时间步$t$的输入。根据GRU最终隐状态更新公式：
$$H_t = Z_t \odot H_{t-1} + (1-Z_t) \odot \tilde{H}_t$$
要让$H_t = H_{t-1}$（即完全保留过去状态信息），需要设置 **更新门 $Z_t=1$**，使得新隐状态完全复制旧隐状态，当前输入被忽略。

对于重置门 $R_t$，由于候选隐状态被乘以$(1-Z_t)=0$而不再影响最终输出，$R_t$的值不影响结果。但从安全角度，设置 **重置门 $R_t=0$** 可以显式阻止候选隐状态中混入当前输入信息。

因此，只想使用时间步$t'$的输入来预测时间步$t$的输出时，对于每个时间步，最佳值为：**更新门 $Z=1$，重置门 $R=0$**。


### 练习 9.1.2

**题目：** 调整和分析超参数对运行时间、困惑度和输出顺序的影响。

**解答：** 减小迭代周期和学习率会导致模型无法在规定周期内完全收敛，造成困惑度增大；减小批量大小和时间步长会增加训练时间；对于简单数据集，减小隐藏单元数即可在更短时间内收敛。

以下使用 `torch` 编程进行验证：


In [2]:
import torch
from torch import nn
from src.utils import (load_data_time_machine, train_epoch_ch8, predict_ch8,
                       sgd, Timer, try_gpu, RNNModel)

# [迭代周期，隐藏单元数，批量大小，小批量数据时间步数，学习率]
hyper_0 = [500, 512, 35, 32, 1]
hyper_1 = [250, 512, 35, 32, 1]
hyper_2 = [500, 256, 35, 32, 1]
hyper_3 = [500, 512, 10, 32, 1]
hyper_4 = [500, 512, 35, 10, 1]
hyper_5 = [500, 512, 35, 32, 0.01]
hyper_params = [hyper_0, hyper_1, hyper_2, hyper_3, hyper_4, hyper_5]

def train_ch9(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False):
    """训练模型（定义见第8章）"""
    loss = nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
    print(predict('time traveller'))
    return ppl

perplexity, times = [], []
for i in range(len(hyper_params)):
    num_epochs, num_hiddens, batch_size, num_steps, lr = hyper_params[i]
    train_iter, vocab = load_data_time_machine(batch_size, num_steps)
    gru_layer = nn.GRU(len(vocab), num_hiddens)
    model = RNNModel(gru_layer, len(vocab)).to(try_gpu())
    timer = Timer()
    timer.start()
    value = round(train_ch9(model, train_iter, vocab, lr, num_epochs,
                           try_gpu()), 1)
    t = timer.stop()
    times.append(t)
    perplexity.append(value)
    print(f'hyper[{i}]: Timer:{t:.2f}, perplexity:{value:.1f}')

使用 `PyPTO` 编程进行验证：


In [3]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import PyPTOLinear, PyPTOGRU, loss_fn

class PyPTORNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.linear = PyPTOLinear(rnn_layer.hidden_size, vocab_size)
    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).float()
        Y, state = self.rnn(X, state)
        return self.linear(Y.reshape((-1, Y.shape[-1]))), state
    def begin_state(self, batch_size, device):
        return torch.zeros((self.rnn.num_layers, batch_size,
                           self.rnn.hidden_size), device=device)

hyper_0 = [500, 512, 35, 32, 1]
hyper_1 = [250, 512, 35, 32, 1]
hyper_2 = [500, 256, 35, 32, 1]
hyper_3 = [500, 512, 10, 32, 1]
hyper_4 = [500, 512, 35, 10, 1]
hyper_5 = [500, 512, 35, 32, 0.01]
hyper_params = [hyper_0, hyper_1, hyper_2, hyper_3, hyper_4, hyper_5]

for i, (num_epochs, num_hiddens, bs, nsteps, lr) in enumerate(hyper_params):
    train_iter, vocab = load_data_time_machine(bs, nsteps)
    gru_layer = PyPTOGRU(len(vocab), num_hiddens)  # 默认单层（PyPTOGRU 支持多层）
    net = PyPTORNNModel(gru_layer, len(vocab)).to(device)
    # JIT warmup
    X_w, y_w = next(iter(train_iter))
    s_w = net.begin_state(bs, device)
    l_w = loss_fn(net(X_w.to(device), s_w)[0],
                  y_w.T.reshape(-1).to(device), len(vocab))
    l_w.backward(); net.zero_grad()
    print(f'hyper[{i}]: num_hiddens={num_hiddens}, '
          f'num_epochs={num_epochs}')
    train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_plot=False, verbose=False,
              loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

hyper[0]: num_hiddens=512, num_epochs=500


困惑度 1.1, 5411.0 词元/秒 npu:0


time traveller proceeded anyreal body must have extension in fou
traveller proceeded anyreal body must have extension in fou


hyper[1]: num_hiddens=512, num_epochs=250


困惑度 5.4, 5818.5 词元/秒 npu:0


...

time traveller came back andfilby s anecdote collapsedthe thing 
traveller atw enat ucand ret if a line and that line theref


hyper[5]: num_hiddens=512, num_epochs=500


困惑度 17.5, 5447.2 词元/秒 npu:0


time traveller                                                  


traveller                                                  


### 练习 9.1.3

**题目：** 比较`rnn.RNN`和`rnn.GRU`的不同实现对运行时间、困惑度和输出字符串的影响。

**解答：** GRU 在重置门和更新门的计算中增加了较多矩阵运算，单步计算量更大，训练时间增加；但 GRU 的门控结构能更好的捕捉长期依赖，困惑度通常低于 RNN。

以下使用 `torch` 编程进行验证：


In [4]:
from src.utils import load_data_time_machine, train_ch8, try_gpu, RNNModel

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

# RNN
num_hiddens = 256
rnn_layer = nn.RNN(len(vocab), num_hiddens)
device_d2l = try_gpu()
net_rnn = RNNModel(rnn_layer, vocab_size=len(vocab)).to(device_d2l)
num_epochs, lr = 500, 1
train_ch8(net_rnn, train_iter, vocab, lr, num_epochs, device_d2l,
          use_plot=False)

# GRU
gru_layer = nn.GRU(len(vocab), num_hiddens)
net_gru = RNNModel(gru_layer, len(vocab)).to(device_d2l)
train_ch8(net_gru, train_iter, vocab, lr, num_epochs, device_d2l,
          use_plot=False)

使用 `PyPTO` 编程进行验证：


In [5]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import PyPTOLinear, PyPTOGRU, PyPTORNN, loss_fn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

class PyPTORNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size):
        super().__init__()
        self.rnn = rnn_layer
        self.linear = PyPTOLinear(rnn_layer.hidden_size, vocab_size)
        self.vocab_size = vocab_size
    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).float()
        Y, state = self.rnn(X, state)
        return self.linear(Y.reshape((-1, Y.shape[-1]))), state
    def begin_state(self, batch_size, device):
        return torch.zeros((self.rnn.num_layers, batch_size,
                           self.rnn.hidden_size), device=device)

# RNN
rnn_layer_pypto = PyPTORNN(len(vocab), 256)  # 默认单层（PyPTORNN 仅支持单层）
net_rnn_pypto = PyPTORNNModel(rnn_layer_pypto, len(vocab)).to(device)
X_w, y_w = next(iter(train_iter))
s_w = net_rnn_pypto.begin_state(batch_size, device)
l_w = loss_fn(net_rnn_pypto(X_w.to(device), s_w)[0],
              y_w.T.reshape(-1).to(device), len(vocab))
l_w.backward(); net_rnn_pypto.zero_grad()
train_ch8(net_rnn_pypto, train_iter, vocab, 1, 500, device,
          use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

# GRU
gru_layer_pypto = PyPTOGRU(len(vocab), 256)  # 默认单层（PyPTOGRU 支持多层）
net_gru_pypto = PyPTORNNModel(gru_layer_pypto, len(vocab)).to(device)
s2 = net_gru_pypto.begin_state(batch_size, device)
l2 = loss_fn(net_gru_pypto(X_w.to(device), s2)[0],
             y_w.T.reshape(-1).to(device), len(vocab))
l2.backward(); net_gru_pypto.zero_grad()
train_ch8(net_gru_pypto, train_iter, vocab, 1, 500, device,
          use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

困惑度 1.3, 18083.7 词元/秒 npu:0


time traveller held in his hand was a glittering wist oney roub 
travellering that id and why go mannet rimetime psome if th


困惑度 1.1, 4884.5 词元/秒 npu:0


time traveller after the pauserequired for the proper assimilati


travelleryou can show black is white by argument said filby


### 练习 9.1.4

**题目：** 仅使用重置门修改 GRU——$R_t$和$Z_t$中只保留$R_t$，$Z_t$替换为$1-Z_t$。将修改后的模型与完整 GRU 的性能进行比较。

**解答：** 仅使用重置门（R-only）时，候选隐状态直接作为新隐状态（$H_t = \tilde{H}_t$），模型失去了更新门对信息的长期记忆能力；仅使用更新门（Z-only）时，模型可以保留/遗忘信息，但无法显式重置旧状态。理论分析表明两种变体都弱于完整 GRU——在 Time Machine 这类简单字符级数据上差异不明显（实测两种变体均可收敛到困惑度 1.0，与完整 GRU 相当），但在需要长期依赖的更复杂任务上，完整 GRU 的门控组合通常更优。下面分别从零实现两种变体进行验证。

以下使用 `torch` 从零实现并验证：



In [6]:
def get_params_R(vocab_size, num_hiddens, device):
    """仅重置门（R-only）：参数为 W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q"""
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    W_xr, W_hr, b_r = three()
    W_xh, W_hh, b_h = three()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def gru_reset_only(inputs, state, params):
    W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        R = torch.sigmoid((X @ W_xr) + (H @ W_hr) + b_r)
        H_tilde = torch.tanh((X @ W_xh) + ((R * H) @ W_hh) + b_h)
        H = H_tilde  # 无更新门：隐状态直接取候选隐状态
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

以下定义仅更新门（Z-only）的参数获取函数：


In [7]:
def get_params_Z(vocab_size, num_hiddens, device):
    """仅更新门（Z-only）：参数为 W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q"""
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    W_xz, W_hz, b_z = three()
    W_xh, W_hh, b_h = three()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def gru_update_only(inputs, state, params):
    W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = torch.sigmoid((X @ W_xz) + (H @ W_hz) + b_z)
        H_tilde = torch.tanh((X @ W_xh) + (H @ W_hh) + b_h)
        H = Z * H + (1 - Z) * H_tilde  # 无重置门：候选隐状态使用旧 H 计算
        Y = H @ W_hq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

使用 `PyPTO` 编程实现仅重置门 / 仅更新门的 GRU：


In [2]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import (PyPTOMatmul, PyPTOBiasAdd, PyPTOAdd,
                           PyPTOTanh, PyPTOSigmoid, PyPTOMul, PyPTOSub,
                           loss_fn)

def get_params_R_pypto(vocab_size, num_hiddens, device):
    """仅重置门（R-only）：W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q"""
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    W_xr, W_hr, b_r = three()
    W_xh, W_hh, b_h = three()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def gru_reset_only_pypto(inputs, state, params):
    """仅重置门 GRU 前向：H_t = H_tilde_t（无更新门）"""
    W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        R = PyPTOSigmoid.apply(PyPTOBiasAdd.apply(
            PyPTOAdd.apply(PyPTOMatmul.apply(X, W_xr),
                           PyPTOMatmul.apply(H, W_hr)), b_r))
        H_tilde = PyPTOTanh.apply(PyPTOBiasAdd.apply(
            PyPTOAdd.apply(PyPTOMatmul.apply(X, W_xh),
                           PyPTOMatmul.apply(PyPTOMul.apply(R, H), W_hh)), b_h))
        H = H_tilde
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

def get_params_Z_pypto(vocab_size, num_hiddens, device):
    """仅更新门（Z-only）：W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q"""
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    W_xz, W_hz, b_z = three()
    W_xh, W_hh, b_h = three()
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def gru_update_only_pypto(inputs, state, params):
    """仅更新门 GRU 前向：H_t = Z_t*H_{t-1} + (1-Z_t)*H_tilde_t（无重置门）"""
    W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = PyPTOSigmoid.apply(PyPTOBiasAdd.apply(
            PyPTOAdd.apply(PyPTOMatmul.apply(X, W_xz),
                           PyPTOMatmul.apply(H, W_hz)), b_z))
        H_tilde = PyPTOTanh.apply(PyPTOBiasAdd.apply(
            PyPTOAdd.apply(PyPTOMatmul.apply(X, W_xh),
                           PyPTOMatmul.apply(H, W_hh)), b_h))
        ones = torch.ones_like(Z)
        one_minus_z = PyPTOSub.apply(ones, Z)
        H = PyPTOAdd.apply(PyPTOMul.apply(Z, H),
                           PyPTOMul.apply(one_minus_z, H_tilde))
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

class PyPTORNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device, get_params,
                 init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn
    def __call__(self, X, state):
        X = F.one_hot(X.T.long(), self.vocab_size).float()
        return self.forward_fn(X, state, self.params)
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad[:] = 0

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

init_state_fn = lambda b, h, d: (torch.zeros((b, h), device=d),)

# ── R-only：warmup + 训练 ──
net_w_r = PyPTORNNModelScratch(len(vocab), 256, device, get_params_R_pypto,
                               init_state_fn, gru_reset_only_pypto)
X_w, y_w = next(iter(train_iter))
s_w = net_w_r.begin_state(batch_size, device)
l = loss_fn(net_w_r(X_w.to(device), s_w)[0],
            y_w.T.reshape(-1).to(device), len(vocab))
l.backward()

net_r = PyPTORNNModelScratch(len(vocab), 256, device, get_params_R_pypto,
                             init_state_fn, gru_reset_only_pypto)
print('仅重置门（R-only）训练：')
train_ch8(net_r, train_iter, vocab, 1, 500, device, use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

# ── Z-only：warmup + 训练 ──
net_w_z = PyPTORNNModelScratch(len(vocab), 256, device, get_params_Z_pypto,
                               init_state_fn, gru_update_only_pypto)
s_w = net_w_z.begin_state(batch_size, device)
l = loss_fn(net_w_z(X_w.to(device), s_w)[0],
            y_w.T.reshape(-1).to(device), len(vocab))
l.backward()

net_z = PyPTORNNModelScratch(len(vocab), 256, device, get_params_Z_pypto,
                             init_state_fn, gru_update_only_pypto)
print('仅更新门（Z-only）训练：')
train_ch8(net_z, train_iter, vocab, 1, 500, device, use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

仅重置门（R-only）训练：


困惑度 1.0, 7209.1 词元/秒 npu:0


time travelleryou can show black is white by argument said filby
travelleryou can show black is white by argument said filby


仅更新门（Z-only）训练：


困惑度 1.0, 5925.7 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e
travelleryou can show black is white by argument said filby


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
